## Variáveis de ambiente (3) — Wyscout

`download_data.py` e `download_heatmaps.py` usam os mesmos três valores:

- `WYSCOUT_SEARCH_TOKEN` — token da API (query param `token`)
- `WYSCOUT_GROUP_ID` — ex.: `1432001`
- `WYSCOUT_SUBGROUP_ID` — ex.: `479792`

### No terminal (zsh/bash)

```bash
export WYSCOUT_SEARCH_TOKEN="…"
export WYSCOUT_GROUP_ID="…"
export WYSCOUT_SUBGROUP_ID="…"
```

### Neste notebook

Preenche os valores na célula seguinte **antes** de correr o download.

In [42]:
import os
from pathlib import Path


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "download_data.py").is_file():
            return p
    raise FileNotFoundError(
        "Não encontrei a raiz do repo (scripts/download_data.py). "
        "Abre o notebook a partir de raumdeuterapp ou faz cd para essa pasta."
    )


ROOT = find_repo_root()
os.chdir(ROOT)
print("Repo root:", ROOT)

# Preencher (ou já exportadas no shell antes de iniciar Jupyter)
os.environ["WYSCOUT_SEARCH_TOKEN"] = "ac7f9db395bc1fcb766500d12fcf0c6caef9bf01"
os.environ["WYSCOUT_GROUP_ID"] = "1432001"
os.environ["WYSCOUT_SUBGROUP_ID"] = "479792"

Repo root: /Users/fbobiano/Projects/raumdeuterappv2


## 1. Download — `download_data.py`

Descarrega resultados da pesquisa Wyscout para `data/players/wyscout/` (um CSV por liga/temporada). Há ~2 s entre pedidos HTTP.

In [14]:
%run scripts/download_data.py

Fetching Premier League (competition=8, season=-6963 / 2025/2026) -> Premier League 25-26.csv
  page 0: got 500 new rows (+0 dups) (total so far: 500) | page_current=0 page_count=2 next=yes
  page 1 attempt 1: 5 dup(s) — retrying in 6s...
  page 1 attempt 2: 5 dup(s) — retrying in 6s...
  page 1 attempt 3: 5 dup(s) — retrying in 6s...
  page 1: got 33 new rows (+0 dups) (total so far: 533) | page_current=1 page_count=2 next=yes
  wrote 533 rows
Fetching Serie A (competition=13, season=-6964 / 2025/2026) -> Serie A 25-26.csv
  page 0: got 500 new rows (+0 dups) (total so far: 500) | page_current=0 page_count=2 next=yes
  page 1 attempt 1: 11 dup(s) — retrying in 6s...
  page 1 attempt 2: 18 dup(s) — retrying in 6s...
  page 1 attempt 3: 11 dup(s) — retrying in 6s...
  page 1 attempt 4: 18 dup(s) — retrying in 6s...
  page 1: 11 dup(s) after 5 retries — skipping dups.
  page 1: got 60 new rows (+11 dups) (total so far: 560) | page_current=1 page_count=2 next=yes
  wrote 560 rows
Fetching

## 1.b Heatmaps Wyscout — `download_heatmaps.py`

Opcional: **depois** de existirem os parquets consolidados `data/players/all/{ano}_all_leagues.parquet` (gerados mais abaixo no pipeline). Usa os **mesmos** três tokens que na secção inicial.

Escreve `data/players/heatmaps/heatmaps_2025.parquet` e `heatmaps_2026.parquet` (GraphQL `playerHeatmap`, 5 workers em paralelo; re-correr só preenche pares em falta).

### No terminal (operacional — precisa das variáveis Wyscout)

```bash
WYSCOUT_SEARCH_TOKEN=… WYSCOUT_GROUP_ID=… WYSCOUT_SUBGROUP_ID=… \
  apps/api/.venv/bin/python scripts/download_heatmaps.py
```

Substitui `…` pelos valores (ou usa `export` nas três linhas e um `apps/api/.venv/bin/python scripts/download_heatmaps.py` sem prefixo).

### A partir deste notebook

Corre a célula seguinte **depois** da célula que define `ROOT` e `os.environ["WYSCOUT_*"]`; usa o Python da venv `apps/api` (onde estão `pandas` / `pyarrow`).

In [43]:
import subprocess

venv_python = ROOT / "apps" / "api" / ".venv" / "bin" / "python"
script = ROOT / "scripts" / "download_heatmaps.py"
if not venv_python.is_file():
    raise FileNotFoundError(
        f"Falta {venv_python} — na raiz do repo: cd apps/api && uv sync"
    )
subprocess.run([str(venv_python), str(script)], check=True, cwd=ROOT)


=== season 2025 ===
  source: /Users/fbobiano/Projects/raumdeuterappv2/data/players/all/2025_all_leagues.parquet
  output: /Users/fbobiano/Projects/raumdeuterappv2/data/players/heatmaps/heatmaps_2025.parquet
  WARN: 8299 rows with unknown Competition (no competition_id) — skipping. Names: ['1 Lyga', '1. Deild', '1. Division', '1. HNL Juniori', '1. Lig', '1. Liga Classic', '1. Liga Promotion', '1. SNL', '1a Divisió', '1st Division', '2. Division', '2. Lig', '2. Liga', '2. Liga Interregional', '2nd Division', '3. Division', '3. Liga', 'A Lyga', 'A-League', 'A.LeCoq Premium Liiga', 'Abissnet Superiore', 'Allsvenskan U19', 'Arabian Gulf Reserve League', 'Baiano 1', 'Baiano U20', 'Besta-deild karla', 'Botola Pro', 'Brasileiro U17', 'Bölgesel Amatör Lig', 'CAS Elite U19 League', 'CSL', 'Campeonato Nacional U18', 'Campeonato Nacional U20', 'Campionato Nazionale Under 17 A&B', 'Campionato Primavera 1', 'Campionato Primavera 2', 'Campionato Primavera 3 - Dante Berretti', 'Canadian Premier Leag

CompletedProcess(args=['/Users/fbobiano/Projects/raumdeuterappv2/apps/api/.venv/bin/python', '/Users/fbobiano/Projects/raumdeuterappv2/scripts/download_heatmaps.py'], returncode=0)

## 2. Limpeza — `cleaning_data.py`

Lê/escreve UTF-8 em `data/players/wyscout/*.csv` e normaliza células que parecem listas Python (ex.: `['LWF', 'LW']` → texto separado por vírgulas).

In [44]:
%run scripts/cleaning_data.py

1. HNL 15-16.csv: 200 rows
1. HNL 16-17.csv: 197 rows
1. HNL 17-18.csv: 206 rows
1. HNL 18-19.csv: 319 rows
1. HNL 19-20.csv: 328 rows
1. HNL 20-21.csv: 344 rows
1. HNL 21-22.csv: 349 rows
1. HNL 22-23.csv: 350 rows
1. HNL 23-24.csv: 378 rows
1. HNL 24-25.csv: 334 rows
1. HNL 25-26.csv: 329 rows
2. Bundesliga 15-16.csv: 241 rows
2. Bundesliga 16-17.csv: 300 rows
2. Bundesliga 17-18.csv: 304 rows
2. Bundesliga 18-19.csv: 456 rows
2. Bundesliga 19-20.csv: 484 rows
2. Bundesliga 20-21.csv: 493 rows
2. Bundesliga 21-22.csv: 507 rows
2. Bundesliga 22-23.csv: 496 rows
2. Bundesliga 23-24.csv: 494 rows
2. Bundesliga 24-25.csv: 523 rows
2. Bundesliga 25-26.csv: 538 rows
Allsvenskan 2015.csv: 151 rows
Allsvenskan 2016.csv: 187 rows
Allsvenskan 2017.csv: 221 rows
Allsvenskan 2018.csv: 373 rows
Allsvenskan 2019.csv: 388 rows
Allsvenskan 2020.csv: 401 rows
Allsvenskan 2021.csv: 410 rows
Allsvenskan 2022.csv: 423 rows
Allsvenskan 2023.csv: 454 rows
Allsvenskan 2024.csv: 455 rows
Allsvenskan 2025.cs

## 2.5. Enriquecimento Transfermarkt — `enrich_with_tm.py`

Corre **depois** da limpeza (passo 2). Usa `data/tm/people.csv` e `data/tm/players.csv` para juntar `image_url` e altura (`height_in_cm` → coluna `Height`) aos CSV Wyscout em `data/players/wyscout/` (sobrescreve in-place).

- Primeira execução: `--dry-run` (preview; não grava).
- Segunda execução: sem `--dry-run` para gravar.

In [45]:
%run ./scripts/enrich_with_tm.py 

Loading TM mapping…
  141,183 distinct player ids (Wyscout-id + Soccerway-id) with TM rows
Enriching 450 wyscout CSVs…
  1. HNL 15-16.csv: rows=200 image_url_total=119 height_filled=200
  1. HNL 16-17.csv: rows=197 image_url_total=121 height_filled=197
  1. HNL 17-18.csv: rows=206 image_url_total=125 height_filled=206
  1. HNL 18-19.csv: rows=319 image_url_total=309 height_filled=319
  1. HNL 19-20.csv: rows=328 image_url_total=313 height_filled=328
  1. HNL 20-21.csv: rows=344 image_url_total=329 height_filled=344
  1. HNL 21-22.csv: rows=349 image_url_total=312 height_filled=349
  1. HNL 22-23.csv: rows=350 image_url_total=320 height_filled=350
  1. HNL 23-24.csv: rows=378 image_url_total=323 height_filled=378
  1. HNL 24-25.csv: rows=334 image_url_total=266 height_filled=334
  1. HNL 25-26.csv: rows=329 image_url_total=232 height_filled=329
  2. Bundesliga 15-16.csv: rows=241 image_url_total=190 height_filled=241
  2. Bundesliga 16-17.csv: rows=300 image_url_total=207 height_filled=

## 3. New performance Index — `new_performance_index.py`


In [46]:
%run  ./transformation/new_performance_index.py

⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Austrian Bundled esliga (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chilean Primera División (not in ALLOWED_LEAGUES)
⚠️ Skipping Chil

## 4. CSV → Parquet — `csv_to_parquet.py`

Converte cada `*_all_leagues.csv` em `data/players/all/` para `.parquet` (snappy). Os CSV agregados por época precisam de já existir nessa pasta (se os geras noutro script, corre esse passo antes deste).

In [47]:
%run ./scripts/csv_to_parquet.py

  2015_all_leagues.csv -> 2015_all_leagues.parquet ... 19.9 MB -> 6.4 MB  (32%), CSV removed
  2016_all_leagues.csv -> 2016_all_leagues.parquet ... 26.6 MB -> 8.2 MB  (31%), CSV removed
  2017_all_leagues.csv -> 2017_all_leagues.parquet ... 28.7 MB -> 8.8 MB  (31%), CSV removed
  2018_all_leagues.csv -> 2018_all_leagues.parquet ... 43.0 MB -> 12.9 MB  (30%), CSV removed
  2019_all_leagues.csv -> 2019_all_leagues.parquet ... 43.6 MB -> 13.0 MB  (30%), CSV removed
  2020_all_leagues.csv -> 2020_all_leagues.parquet ... 47.3 MB -> 13.9 MB  (29%), CSV removed
  2021_all_leagues.csv -> 2021_all_leagues.parquet ... 51.4 MB -> 15.0 MB  (29%), CSV removed
  2022_all_leagues.csv -> 2022_all_leagues.parquet ... 50.7 MB -> 14.8 MB  (29%), CSV removed
  2023_all_leagues.csv -> 2023_all_leagues.parquet ... 53.0 MB -> 15.3 MB  (29%), CSV removed
  2024_all_leagues.csv -> 2024_all_leagues.parquet ... 49.2 MB -> 14.2 MB  (29%), CSV removed
  2025_all_leagues.csv -> 2025_all_leagues.parquet ... 54.2 MB 

## 5. Update potential scores— `train_potential.py`

In [48]:
%run scripts/train_potential.py

Loading merged season files …
Rows: 172,845, seasons: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Enriching (league power, position type, performance index) …
Computing target: within 3 seasons, strong league (power≥85) and performance_index>73 …
Training-eligible rows (age≤23, minutes≥450, position ok, future known): 25,840
Positive rate (strong league + PI>73): 0.065
Temporal hold-out: train season_year < 2023
Validation metrics: {
  "n_train": 18194.0,
  "n_test": 7646.0,
  "roc_auc": 0.9370241622518156,
  "brier": 0.09888231275599994
}
Fitting final model on all labeled rows …
Saved model → /Users/fbobiano/Projects/raumdeuterappv2/data/potential/potential_model.joblib
Scoring full cohort across all seasons (n=37,913) …
Saved scores → /Users/fbobiano/Projects/raumdeuterappv2/data/potential/potential_scores.parquet and /Users/fbobiano/Pr

## check duplicates

In [26]:
%run scripts/check_wyscout_duplicate_players.py --only-with-dups


file                                                        rows   dup_keys  excess_rows  notes
-----------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
Files scanned: 450  With duplicate keys: 0  Sum of duplicate_key counts: 0


## Remover duplicados nos CSV Wyscout — `dedupe_wyscout_csv_rows.py`

Por ficheiro em `data/players/wyscout/`, remove linhas repetidas com a mesma chave **(Wyscout id, nome do jogador, equipa)**. Por omissão mantém a **última** ocorrência (`--keep last`); usa `--keep first` ou `--dry-run` no terminal conforme precises.

**Ordem lógica:** corre isto logo **após** a limpeza (secção 2) e **antes** do índice de performance (secção 3). Esta célula está no fim do notebook só como referência ao script; sobe-a ou corre-a no momento certo do pipeline.

In [ ]:
%run scripts/dedupe_wyscout_csv_rows.py

## 10. Filtered player valuations — `build_player_valuations.py`

Cria `data/tm/player_valuations_filtered.csv` (e parquet com `--parquet`) com colunas  
`wyscout_id`, `key_transfermarkt`, `date`, `market_value_in_eur`. Por defeito mantém  
qualquer jogador que apareça em pelo menos um ficheiro Wyscout (`--mode union`).

_Para o xTV v2 e `utils.tm_market_value` usa-se o ficheiro bruto **`player_valuations.csv`** — este filtro é opcional (só reduz I/O noutros fluxos)._

_Para uso estrito (jogadores em **todos** os ficheiros — geralmente devolve 0):  
`--mode intersection`._


In [49]:
%run  scripts/build_player_valuations.py --parquet

Scanning Wyscout files for player ids…
  450 files scanned
  52,538 export player ids in union
  25,426 people rows mapped to key_transfermarkt (wyscout + soccerway)
Reading player_valuations.csv…
  369,699 valuation rows after player_id filter
Wrote /Users/fbobiano/Projects/raumdeuterappv2/data/tm/player_valuations_filtered.csv (369,699 rows)
Wrote /Users/fbobiano/Projects/raumdeuterappv2/data/tm/player_valuations_filtered.parquet


## 11. Club logos parquet — `build_club_logos.py`

Percorre todos os CSVs Wyscout e produz `data/teams/club_logos.parquet` com colunas **`team`**, **`logo_url`**, **`competition`** (doméstica; alinha com `league` nos parquets), **`n_seasons`**, **`latest_file`**. Inclui todas as equipas (sem filtro). Se o CSV não tiver coluna `Competition`, fica string vazia `""`.

In [ ]:
%run build_club_logos.py

## 12. Resumo — ordem sugerida de scripts

1. **`download_data.py`** — CSV Wyscout por liga/temporada em `data/players/wyscout/`.
2. **`download_heatmaps.py`** — heatmaps TM (venv API; mesmo env Wyscout das células iniciais).
3. **`cleaning_data.py`** — limpezas/fixes antes do pipeline estatístico.
4. **`enrich_with_tm.py`** — foto/altura desde `people.csv` + `players.csv`.
5. **`transformation/new_performance_index.py`** — gera `{ano}_all_leagues.csv`.
6. **`csv_to_parquet.py`** — `data/players/all/{ano}_all_leagues.parquet` (consumo pela API DuckDB).
7. **`train_potential.py`** — scores de potencial (parquet esperado pela API).
8. **`check_wyscout_duplicate_players.py`** + **`dedupe_wyscout_csv_rows.py`** — auditoria/remoção de duplicados nos CSV Wyscout antes de novo ciclo opcional de (5–6).
9. **`build_player_valuations.py --parquet`** — subset TM filtrado por jogadores Wyscout (`player_valuations_filtered`).
10. **`build_club_logos.py`** — `club_logos.parquet` (`team`, `logo_url`, `competition`).
11. **`train_xtv.py`** — modelo xTV v2 (transferências TM × parquets Wyscout; precisa `data/tm/transfers.csv`, `people.csv`, `players.csv`, `player_valuations.csv`, `clubs.csv`, parquets em `data/players/all/`). Grava **`models/xtv_v2.joblib`**, **`models/xtv_v2.json`**, **`data/tm/xtv_id_mapping.parquet`** e prior de destino em `data/tm/` (venv com sklearn/joblib/scipy).
12. **`apply_xtv_to_parquets.py`** — acrescenta **`tm_market_value_eur`** e **`x_tv_eur`** nos parquets (por defeito **sobrescreve** `{ano}_all_leagues.parquet`; `--sidecar` grava `*_xtv.parquet` à parte).

_Dependências de dados_: TM (`data/tm/…`) atualizadas por ingestão própria; este notebook não faz scrape TM.

## 13. xTV v2 — treino (`train_xtv.py`)

Junta **`transfers.csv`** aos parquets **`{ano}_all_leagues.parquet`**, mapeia IDs Wyscout↔TM (`xtv_id_mapping.parquet`) e usa **`player_valuations.csv`** (MV as-of na data da transferência). Treina duas cabeças (ratio `log(fee/mv)` quando há MV; `log(fee)` absoluto) com split temporal em **`--cutoff`** (default `2023-07-01`); no fim refit no dataset completo.

- Saídas: **`models/xtv_v2.joblib`** + **`models/xtv_v2.json`** (métricas), **`data/tm/xtv_id_mapping.parquet`**, prior de destino em `data/tm/`.
- Argumentos úteis: `--max-rows N` (debug), `--fuzzy-threshold 92`, `--no-refresh-mappings` (reutilizar mapping existente).

Usa o **mesmo Python do venv da API** (sklearn, joblib, scipy).

In [34]:
import subprocess
venv_python = ROOT / "apps" / "api" / ".venv" / "bin" / "python"
script = ROOT / "scripts" / "train_xtv.py"
if not venv_python.is_file():
    raise FileNotFoundError(
        f"Falta {venv_python} — na raiz: cd apps/api && uv sync",
    )
subprocess.run(
    [
        str(venv_python),
        str(script),
        # "--cutoff", "2023-07-01",
        # "--max-rows", "8000",
        # "--no-refresh-mappings",
    ],
    check=True,
    cwd=ROOT,
)

Traceback (most recent call last):
  File "/Users/fbobiano/Projects/raumdeuterappv2/scripts/train_xtv.py", line 160, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/Users/fbobiano/Projects/raumdeuterappv2/scripts/train_xtv.py", line 28, in main
    from transformation.xtv import (
    ...<6 lines>...
    )
ModuleNotFoundError: No module named 'transformation.xtv'


CalledProcessError: Command '['/Users/fbobiano/Projects/raumdeuterappv2/apps/api/.venv/bin/python', '/Users/fbobiano/Projects/raumdeuterappv2/scripts/train_xtv.py']' returned non-zero exit status 1.

## 14. xTV v2 — aplicar aos parquets (`apply_xtv_to_parquets.py`)

Lê **`models/xtv_v2.joblib`**, acrescenta **`tm_market_value_eur`** (MV TM as-of fim de época; imputação por peers se em falta) e **`x_tv_eur`** (média de previsões com destinos amostrados do prior de treino).

- Por defeito **sobrescreve** `{ano}_all_leagues.parquet` (o que a API DuckDB lê).
- **`--sidecar`** grava `{ano}_all_leagues_xtv.parquet` sem tocar no original.
- **`--dry-run`** lista ficheiros sem gravar.

In [50]:
import subprocess

venv_python = ROOT / "apps" / "api" / ".venv" / "bin" / "python"
script = ROOT / "scripts" / "apply_xtv_to_parquets.py"
if not venv_python.is_file():
    raise FileNotFoundError(
        f"Falta {venv_python} — na raiz: cd apps/api && uv sync",
    )

subprocess.run(
    [
        str(venv_python),
        str(script),
        "--model",
        str(ROOT / "models" / "xtv_v2.joblib"),
        # "--sidecar",             # grava *_all_leagues_xtv.parquet em vez de overwrite
        # "--dry-run",             # imprime apenas o que corraria
    ],
    check=True,
    cwd=ROOT,
)


Building peer MV table …
  peer buckets: 0
  2015_all_leagues.parquet: xTV 7,526/7,526 non-null, TM mv 5,536/7,526 non-null
  2016_all_leagues.parquet: xTV 10,177/10,177 non-null, TM mv 6,946/10,177 non-null
  2017_all_leagues.parquet: xTV 10,796/10,796 non-null, TM mv 7,582/10,796 non-null
  2018_all_leagues.parquet: xTV 16,449/16,449 non-null, TM mv 11,033/16,449 non-null
  2019_all_leagues.parquet: xTV 16,621/16,621 non-null, TM mv 11,283/16,621 non-null
  2020_all_leagues.parquet: xTV 18,160/18,160 non-null, TM mv 12,342/18,160 non-null
  2021_all_leagues.parquet: xTV 19,757/19,757 non-null, TM mv 13,242/19,757 non-null
  2022_all_leagues.parquet: xTV 19,317/19,317 non-null, TM mv 13,388/19,317 non-null
  2023_all_leagues.parquet: xTV 20,088/20,088 non-null, TM mv 13,702/20,088 non-null
  2024_all_leagues.parquet: xTV 18,815/18,815 non-null, TM mv 13,619/18,815 non-null
  2025_all_leagues.parquet: xTV 20,712/20,712 non-null, TM mv 14,035/20,712 non-null
  2026_all_leagues.parquet: 

CompletedProcess(args=['/Users/fbobiano/Projects/raumdeuterappv2/apps/api/.venv/bin/python', '/Users/fbobiano/Projects/raumdeuterappv2/scripts/apply_xtv_to_parquets.py', '--model', '/Users/fbobiano/Projects/raumdeuterappv2/models/xtv_v2.joblib'], returncode=0)